#Prepare Environment

In [ ]:
pip install torch openai-whisper jiwer librosa

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.5/800.5 kB 8.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 113.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 52.2 MB/

In [ ]:
import torch
import whisper
import jiwer
import os
import librosa
import pandas as pd

#Load&prepare Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
test_audio_folder = "/content/drive/MyDrive/L2-KSU/dataset"

csv_path = "/content/drive/MyDrive/L2-KSU/ l2-ksu-test.csv"
df = pd.read_csv(csv_path)

# Load the Whisper small model

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = whisper.load_model("small").to(device)

100%|███████████████████████████████████████| 461M/461M [00:07<00:00, 63.3MiB/s]
/usr/local/lib/python3.11/dist-packages/whisper/__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this exper

In [ ]:
# Create ground truth dictionary
ground_truths = dict(zip(df["path"], df["text"]))

# Function to transcribe audio using Whisper
def transcribe_audio(file_path):
    audio, _ = librosa.load(file_path, sr=16000)  # Load audio with 16kHz sample rate
    result = model.transcribe(file_path,language='ar')  # Transcribe using Whisper
    return result["text"].lower().strip()  # Convert to lowercase for consistency


In [ ]:
# Compute WER
wer_scores = []
for file_name, true_text in ground_truths.items():
    audio_path = os.path.join(test_audio_folder, file_name)

    if not os.path.exists(audio_path):
        print(f"Warning: {audio_path} not found!")
        continue  # Skip missing files

    print(f"Processing: {file_name}...")

    # Transcribe
    predicted_text = transcribe_audio(audio_path)

    # Compute WER for this sample
    wer = jiwer.wer(true_text, predicted_text)
    wer_scores.append(wer)

    # Print intermediate results
    print(f"Ground Truth: {true_text}")
    print(f"Predicted: {predicted_text}")
    print(f"WER: {wer:.4f}\n")



Processing: /content/drive/MyDrive/L2-KSU/dataset/non-native_speakers/males/speaker2/sentence9/wav/Sp2_M_S9_2.wav...
Ground Truth: صلى الله على نبينا محمد و على آله و صحبه أجمعين
Predicted: وصلى الله على نبينا محبن وعالى آل وصحب يجمعين
WER: 0.7273

Processing: /content/drive/MyDrive/L2-KSU/dataset/non-native_speakers/females/speaker1/sentence10/wav/Sp1_F_S10_3.wav...
Ground Truth: سبحان الله و بحمده سبحان الله العظيم
Predicted: سبحان الله وبحمده سبحان الله الآعادي
WER: 0.4286

Processing: /content/drive/MyDrive/L2-KSU/dataset/non-native_speakers/females/speaker1/sentence3/wav/Sp1_F_S3_3.wav...
Ground Truth: صراط الذين أنعمت عليهم
Predicted: سرات الذين أنعمت عليهم
WER: 0.2500

Processing: /content/drive/MyDrive/L2-KSU/dataset/non-native_speakers/males/speaker3/sentence1/wav/Sp3_M_S1_2.wav...
Ground Truth: الحمد لله رب العالمين الرحمن الرحيم
Predicted: الحمد لله رب العالمين الرحمن روحيم
WER: 0.1667

Processing: /content/drive/MyDrive/L2-KSU/dataset/non-native_speakers/females/speaker2/se

In [ ]:
# Calculate average WER
if wer_scores:
    baseline_wer = sum(wer_scores) / len(wer_scores)
    print(f" WER before fine-tuning: {baseline_wer:.4f}")
else:
    print("No valid test files found.")

\ WER before fine-tuning: 0.5305


## Load the Whisper model with different sizes

In [ ]:
model_sizes = ["tiny", "base","small", "medium", "large", "turbo"]

for size in model_sizes:
    print(f"\nRunning Whisper {size} model...")
    model = whisper.load_model(size).to(device)

    wer_scores = []
    for audio_path, true_text in ground_truths.items():
        predicted_text = model.transcribe(audio_path, language="ar")["text"]

        wer = jiwer.wer(true_text, predicted_text)
        wer_scores.append(wer)

    baseline_wer = sum(wer_scores) / len(wer_scores)
    print(f" WER for {size}: {baseline_wer:.4f}")



Running Whisper tiny model...
 WER for tiny: 0.7831

Running Whisper base model...


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 183MiB/s]
/usr/local/lib/python3.11/dist-packages/whisper/__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this exper

 WER for base: 0.6919

Running Whisper small model...


/usr/local/lib/python3.11/dist-packages/whisper/__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(fp, map_location=device)


 WER for small: 0.5146

Running Whisper medium model...


...

 WER for medium: 0.3944

Running Whisper large model...


100%|█████████████████████████████████████| 2.88G/2.88G [00:43<00:00, 71.8MiB/s]
/usr/local/lib/python3.11/dist-packages/whisper/__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this exper

 WER for large: 0.3432

Running Whisper turbo model...


100%|█████████████████████████████████████| 1.51G/1.51G [00:39<00:00, 41.5MiB/s]
/usr/local/lib/python3.11/dist-packages/whisper/__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this exper

 WER for turbo: 0.3638
